# Saved checkpoint semantic diagnostic

Select a CPU runtime. When Colab prompts for a file, upload the matching `colab_distillation_bundle.zip`. The first cell mounts Drive and runs the offline preflight. Review its report before running the paid cell.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path
from google.colab import drive, files
import hashlib
import zipfile

from io import BytesIO
from tempfile import TemporaryDirectory, mkdtemp


def verified_project(project, bundle, checksum):
    payload = bundle.read_bytes() if bundle.is_file() else b""
    if hashlib.sha256(payload).hexdigest() != checksum:
        raise ValueError("Diagnostic ZIP checksum does not match")
    with zipfile.ZipFile(BytesIO(payload)) as archive:
        expected = {}
        for member in archive.infolist():
            path = Path(member.filename)
            if (path.is_absolute() or ".." in path.parts or len(path.parts) < 2
                    or path.parts[0] != project.name or member.is_dir()):
                raise ValueError("Unexpected diagnostic bundle member")
            relative = Path(*path.parts[1:])
            if relative in expected:
                raise ValueError("Duplicate diagnostic bundle member")
            expected[relative] = hashlib.sha256(archive.read(member)).hexdigest()

        def matches(directory):
            if not directory.is_dir() or directory.is_symlink():
                return False
            paths = list(directory.rglob("*"))
            if any(path.is_symlink() for path in paths):
                return False
            files = {path.relative_to(directory) for path in paths if path.is_file()}
            return files == expected.keys() and all(
                hashlib.sha256((directory / file).read_bytes()).hexdigest() == digest
                for file, digest in expected.items())

        if matches(project):
            return project
        with TemporaryDirectory(dir=project.parent) as folder:
            fresh = Path(folder) / project.name
            archive.extractall(folder)
            if not matches(fresh):
                raise ValueError("Extracted diagnostic code does not match the verified bundle")
            previous = None
            if project.exists() or project.is_symlink():
                previous = Path(mkdtemp(prefix=".stale-diagnostic-", dir=project.parent)) / project.name
                project.replace(previous)
            try:
                fresh.replace(project)
            except OSError:
                if previous is not None:
                    previous.replace(project)
                raise
        if not matches(project):
            raise ValueError("Installed diagnostic code failed verification")
        return project


drive.mount("/content/drive")
project = Path("/content/Health-Insurance-SLM-Colab")
bundle = Path("/content/colab_distillation_bundle.zip")
EXPECTED_BUNDLE_SHA256 = "f17374970dbd64ea14b7ca644cd0d437dd38b2f68b4f1d532319926a26631c43"
if not bundle.is_file() or hashlib.sha256(bundle.read_bytes()).hexdigest() != EXPECTED_BUNDLE_SHA256:
    uploaded = files.upload()
    payload = next((data for data in uploaded.values()
                    if hashlib.sha256(data).hexdigest() == EXPECTED_BUNDLE_SHA256), None)
    if payload is None:
        raise ValueError("Upload the matching diagnostic code bundle")
    bundle.write_bytes(payload)
project = verified_project(project, bundle, EXPECTED_BUNDLE_SHA256)
root = Path("/content/drive/MyDrive/HealthInsuranceSLM")
original = root / "colab_l4_balanced_5x200_v3"
saved = root / "colab_l4_balanced_5x200_v3_checkpoint_eval"
output = root / "colab_l4_balanced_5x200_v3_semantic_diagnostic"
os.environ.setdefault("DISTILLATION_LOCAL_LOCK_DIR", "/content/distillation-eval-locks")
command = [sys.executable, "-E", "-s", "-B", "-m", "distillation_loop.semantic_eval",
           "--comparison-dir", str(saved), "--split", str(original / "evaluation_split.json"),
           "--checkpoint-root", str(original), "--historical-run", str(original),
           "--output", str(output)]
cache = original / "benchmark_judging"
if cache.is_dir():
    command += ["--legacy-cache", str(cache)]
subprocess.run(command, cwd=project, check=True)

In [ ]:
# Run only after reviewing cost_preflight.json and approving its cumulative cap.
import json
from getpass import getpass

preflight = json.loads((output / "cost_preflight.json").read_text())
if preflight["unique_missing_judgments"]:
    approval = input("Paste the approved preflight hash (blank cancels): ").strip()
    if approval:
        cap = input("Approved cumulative ceiling in USD: ").strip()
        if approval != preflight["approval_hash"]:
            raise ValueError("Approval does not match the preflight")
        env = dict(os.environ)
        if not env.get("OPENAI_API_KEY"):
            env["OPENAI_API_KEY"] = getpass("OpenAI API key: ")
        verified_project(project, bundle, EXPECTED_BUNDLE_SHA256)
        subprocess.run(command + ["--approve-preflight", approval, "--max-usd", cap],
                       cwd=project, env=env, check=True)